# LJ-7 SSPD Experiment

Reproduction of Algorithm 2.1 (Stochastic Saddle Point Dynamics) from:

> T. Lelievre and P. Parpas, *Using Witten Laplacians to Locate Index-1 Saddle Points*,
> SIAM J. Sci. Comput. 46(2), pp. A770-A797, 2024.

Applied to the **7-atom Lennard-Jones cluster in 2D** (Section 4.5 of the paper).

In [ ]:
import sys
sys.path.insert(0, '..')

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

from lj_sspd.potential import make_lj_potential
from lj_sspd.initial_config import load_or_generate
from lj_sspd.sspd import run_sspd, compute_weights
from lj_sspd.langevin import run_langevin
from lj_sspd.utils import check_index1

jax.config.update("jax_enable_x64", True)

# -- Paper-style cluster plotting ----------------------------------------

# Atom colors matching the paper (Tables 4-5, Figs 7-9)
ATOM_COLORS = ['#e6194b', '#3cb44b', '#4363d8', '#f58231',
               '#911eb4', '#42d4f4', '#f032e6', '#bfef45',
               '#fabed4', '#469990', '#dcbeff', '#9A6324']

def plot_cluster(ax, pos, title=None, atom_radius=0.22, bond_cutoff=1.3,
                 show_labels=True, show_bonds=True):
    """Plot an atom cluster in the style of Lelievre & Parpas Tables 4-5."""
    n = pos.shape[0]
    if show_bonds:
        for i in range(n):
            for j in range(i+1, n):
                d = np.linalg.norm(pos[i] - pos[j])
                if d < bond_cutoff:
                    ax.plot([pos[i,0], pos[j,0]], [pos[i,1], pos[j,1]],
                            'k-', lw=1.5, alpha=0.3, zorder=1)
    for i in range(n):
        c = ATOM_COLORS[i % len(ATOM_COLORS)]
        circle = Circle(pos[i], atom_radius, facecolor=c,
                       edgecolor='black', linewidth=1.5, zorder=3)
        ax.add_patch(circle)
        if show_labels:
            ax.text(pos[i,0], pos[i,1], str(i), ha='center', va='center',
                   fontsize=9, fontweight='bold', zorder=4)
    margin = atom_radius + 0.3
    ax.set_xlim(pos[:,0].min() - margin, pos[:,0].max() + margin)
    ax.set_ylim(pos[:,1].min() - margin, pos[:,1].max() + margin)
    ax.set_aspect('equal')
    ax.set_xticks([])
    ax.set_yticks([])
    if title:
        ax.set_title(title, fontsize=11)

def plot_cluster_grid(configs, titles, ncols=4, figsize=None):
    """Plot multiple clusters in a grid (like paper Tables 4-5)."""
    n = len(configs)
    nrows = max(1, int(np.ceil(n / ncols)))
    if figsize is None:
        figsize = (3.2 * ncols, 3.2 * nrows)
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, squeeze=False)
    for i in range(nrows * ncols):
        r, c = divmod(i, ncols)
        ax = axes[r, c]
        if i < n:
            plot_cluster(ax, np.array(configs[i]), title=titles[i])
        else:
            ax.axis('off')
    plt.tight_layout()
    return fig

## 1. Initial Configuration $C_0$

The global minimum of the 7-atom LJ cluster in 2D has energy $\approx -12.535$ (paper Table 4).
This is our starting point for both SSPD and Langevin.

In [ ]:
config = load_or_generate(7, 2, data_dir='../data/')
X0 = config['config']
V_fns = make_lj_potential(N_atoms=7, dim=2)

print(f"C0 energy: {config['energy']:.6f}")
print(f"Gradient norm: {config['grad_norm']:.2e}")

fig, ax = plt.subplots(1, 1, figsize=(4, 4))
plot_cluster(ax, np.array(X0), title=f"$C_0$ (E = {config['energy']:.3f})")
plt.tight_layout()
plt.show()

## 2. Run SSPD (Algorithm 2.1)

Parameters from Table 1 of the paper: $\delta = 10^{-4}$, $\beta^{-1} = 0.1$, $\rho_\text{ess} = 0.99$, $N = 2000$, $K = 10000$.

In [ ]:
sspd_params = {
    'N': 2000, 'K': 10000, 'delta': 1e-4, 'beta_inv': 0.1,
    'rho_ess': 0.99, 'seed': 0, 'stats_interval': 100,
}

result_sspd = run_sspd(V_fns, X0, sspd_params)

In [ ]:
stats = result_sspd['stats']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Panel 1: ESS / N over time
axes[0].plot(stats['step'], stats['ess'], 'k-', lw=1)
axes[0].set_xlabel(r'Iteration $k$', fontsize=11)
axes[0].set_ylabel(r'ESS / $N$', fontsize=11)
axes[0].set_title('Effective Sample Size', fontsize=12)
axes[0].set_ylim(0.95, 1.005)
axes[0].grid(True, alpha=0.3)

# Panel 2: Weight range
axes[1].fill_between(stats['step'], stats['min_w'], stats['max_w'],
                     alpha=0.3, color='steelblue', label='weight range')
axes[1].plot(stats['step'], stats['max_w'], 'b-', lw=1, label=r'$\max_n w_k^n$')
axes[1].plot(stats['step'], stats['min_w'], 'b--', lw=1, label=r'$\min_n w_k^n$')
axes[1].set_xlabel(r'Iteration $k$', fontsize=11)
axes[1].set_ylabel(r'Weight $w_k^n = \Vert Y_k^n \Vert$', fontsize=11)
axes[1].set_title('Particle Weights', fontsize=12)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

# Panel 3: Energy at highest-weight particle
axes[2].plot(stats['step'], stats['max_w_energy'], 'r-', lw=0.8, alpha=0.7)
axes[2].axhline(y=config['energy'], color='k', linestyle='--', lw=1, 
                label=f'$C_0$ energy ({config["energy"]:.3f})')
axes[2].set_xlabel(r'Iteration $k$', fontsize=11)
axes[2].set_ylabel(r'Energy $V(X_k^{n^*})$', fontsize=11)
axes[2].set_title('Energy at Highest-Weight Particle', fontsize=12)
axes[2].legend(fontsize=9)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/sspd_diagnostics.pdf', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Compute final energies for both methods
W_final = np.array(compute_weights(result_sspd['Y_final']))
E_final = np.array(V_fns['V_batch'](result_sspd['X_final']))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Left: energy histogram 
axes[0].hist(E_final, bins=60, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(x=config['energy'], color='red', linestyle='--', lw=1.5, 
                label=f'$C_0$ ({config["energy"]:.3f})')
axes[0].set_xlabel(r'Energy $V(X_K^n)$', fontsize=11)
axes[0].set_ylabel('Count', fontsize=11)
axes[0].set_title('SSPD: Final Particle Energies', fontsize=12)
axes[0].legend(fontsize=10)

# Right: weight vs energy scatter (paper-style blue-red colormap)
w_norm = (W_final - W_final.min()) / (W_final.max() - W_final.min() + 1e-12)
sc = axes[1].scatter(E_final, W_final, c=w_norm, cmap=plt.cm.coolwarm, 
                     s=8, alpha=0.6, edgecolors='none')
axes[1].set_xlabel(r'Energy $V(X_K^n)$', fontsize=11)
axes[1].set_ylabel(r'Weight $w_K^n = \Vert Y_K^n \Vert$', fontsize=11)
axes[1].set_title('SSPD: Weight vs Energy', fontsize=12)
plt.colorbar(sc, ax=axes[1], label='Normalized weight')

plt.tight_layout()
plt.savefig('../data/sspd_final_particles.pdf', bbox_inches='tight', dpi=150)
plt.show()

## 3. Run Langevin (Baseline)

Standard overdamped Langevin dynamics (Eq. 2.2) with the same parameters. 
No $Y$ process, no resampling — exploration by diffusion only.

In [ ]:
lang_params = {
    'N': 2000, 'K': 10000, 'delta': 1e-4, 'beta_inv': 0.1,
    'seed': 0, 'stats_interval': 100,
}

result_lang = run_langevin(V_fns, X0, lang_params)

## 4. SSPD vs Langevin Comparison

Side-by-side comparison of energy exploration and final particle distributions.

In [ ]:
stats_lang = result_lang['stats']
E_final_lang = np.array(V_fns['V_batch'](result_lang['X_final']))

# Backwards-compatible: compute energy stats if not already in SSPD stats
# (needed if run_sspd was called before the mean_energy tracking was added)
if 'mean_energy' not in stats:
    print("Re-computing SSPD energy stats from final particles...")
    stats['mean_energy'] = stats.get('mean_energy', [float(E_final.mean())] * len(stats['step']))
    stats['min_energy'] = stats.get('min_energy', [float(E_final.min())] * len(stats['step']))
    stats['max_energy'] = stats.get('max_energy', [float(E_final.max())] * len(stats['step']))
    print("Note: for accurate time-series, re-run the SSPD cell above (Shift+Enter on cell 5)")

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# --- Row 1: Energy evolution (same metric for both) ---

# Top-left: Mean energy over time
axes[0,0].plot(stats['step'], stats['mean_energy'], 'r-', lw=1, label='SSPD')
axes[0,0].plot(stats_lang['step'], stats_lang['mean_energy'], 'b-', lw=1, label='Langevin')
axes[0,0].axhline(y=config['energy'], color='k', linestyle='--', lw=1, alpha=0.5,
                   label=f'$C_0$ ({config["energy"]:.3f})')
axes[0,0].set_xlabel(r'Iteration $k$', fontsize=11)
axes[0,0].set_ylabel('Mean energy', fontsize=11)
axes[0,0].set_title('Mean Particle Energy', fontsize=12)
axes[0,0].legend(fontsize=9)
axes[0,0].grid(True, alpha=0.3)

# Top-right: Energy range (min-max band) for both
axes[0,1].fill_between(stats['step'], stats['min_energy'], stats['max_energy'],
                       alpha=0.2, color='red', label='SSPD range')
axes[0,1].fill_between(stats_lang['step'], stats_lang['min_energy'], stats_lang['max_energy'],
                       alpha=0.2, color='blue', label='Langevin range')
axes[0,1].plot(stats['step'], stats['mean_energy'], 'r-', lw=1, label='SSPD mean')
axes[0,1].plot(stats_lang['step'], stats_lang['mean_energy'], 'b-', lw=1, label='Langevin mean')
axes[0,1].axhline(y=config['energy'], color='k', linestyle='--', lw=1, alpha=0.5)
axes[0,1].set_xlabel(r'Iteration $k$', fontsize=11)
axes[0,1].set_ylabel('Energy', fontsize=11)
axes[0,1].set_title('Energy Range (min/mean/max)', fontsize=12)
axes[0,1].legend(fontsize=8)
axes[0,1].grid(True, alpha=0.3)

# --- Row 2: Final energy distributions (same bins, same y-scale) ---
bins = np.linspace(min(E_final.min(), E_final_lang.min()),
                   max(E_final.max(), E_final_lang.max()), 60)

axes[1,0].hist(E_final, bins=bins, color='#d62728', edgecolor='white', alpha=0.85)
axes[1,0].axvline(x=config['energy'], color='k', linestyle='--', lw=1.5)
axes[1,0].set_xlabel(r'Energy $V(X_K^n)$', fontsize=11)
axes[1,0].set_ylabel('Count', fontsize=11)
axes[1,0].set_title(r'SSPD: Final Energies ($K=10000$)', fontsize=12)

axes[1,1].hist(E_final_lang, bins=bins, color='#1f77b4', edgecolor='white', alpha=0.85)
axes[1,1].axvline(x=config['energy'], color='k', linestyle='--', lw=1.5)
axes[1,1].set_xlabel(r'Energy $V(X_K^n)$', fontsize=11)
axes[1,1].set_ylabel('Count', fontsize=11)
axes[1,1].set_title(r'Langevin: Final Energies ($K=10000$)', fontsize=12)

# Match y-axis limits for histograms
y_max = max(axes[1,0].get_ylim()[1], axes[1,1].get_ylim()[1])
axes[1,0].set_ylim(0, y_max)
axes[1,1].set_ylim(0, y_max)

plt.tight_layout()
plt.savefig('../data/sspd_vs_langevin.pdf', bbox_inches='tight', dpi=150)
plt.show()

# Summary table
print(f"{'':>12s} {'Mean E':>10s} {'Min E':>10s} {'Max E':>10s} {'Std E':>10s}")
print(f"{'SSPD':>12s} {E_final.mean():>10.3f} {E_final.min():>10.3f} {E_final.max():>10.3f} {E_final.std():>10.3f}")
print(f"{'Langevin':>12s} {E_final_lang.mean():>10.3f} {E_final_lang.min():>10.3f} {E_final_lang.max():>10.3f} {E_final_lang.std():>10.3f}")

## 5. Cluster Analysis: Best SSPD Particle

Visualize the highest-weight particle's configuration in the style of the paper's Tables 4-5. 
Check whether it is in the index-1 region $S = \{x \mid \lambda_1(x) < 0 < \lambda_{2+\text{eig\_gap}}(x)\}$.

In [ ]:
weights = np.array(result_sspd['weights'])
hess_fn = jax.hessian(V_fns['V'])

# Best particle (highest weight)
best_idx = int(np.argmax(weights))
X_best = result_sspd['X_final'][best_idx]
E_best = float(V_fns['V'](X_best))
is_idx1, evals = check_index1(X_best, hess_fn, V_fns['eig_gap'])

# Also show a few top particles
top_k = 5
top_indices = np.argsort(weights)[::-1][:top_k]

# --- Cluster comparison: C0 vs best SSPD particle ---
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
plot_cluster(axes[0], np.array(X0), title=f'$C_0$ (E = {config["energy"]:.3f})')
plot_cluster(axes[1], np.array(X_best), 
             title=f'Best SSPD particle (E = {E_best:.3f})')
plt.tight_layout()
plt.savefig('../data/c0_vs_best.pdf', bbox_inches='tight', dpi=150)
plt.show()

# --- Eigenvalue spectrum ---
evals_np = np.array(evals)
Nd = 7 * 2
eig_gap = V_fns['eig_gap']

fig, ax = plt.subplots(figsize=(8, 3))
colors = ['gray'] * eig_gap + ['red' if evals_np[eig_gap] < -1e-8 else 'green']
colors += ['green'] * (Nd - eig_gap - 1)
# Color: gray = zero modes, red = negative, green = positive
bar_colors = []
for i in range(Nd):
    if i < eig_gap:
        bar_colors.append('lightgray')
    elif evals_np[i] < -1e-8:
        bar_colors.append('#d62728')
    else:
        bar_colors.append('#2ca02c')

ax.bar(range(Nd), evals_np, color=bar_colors, edgecolor='black', linewidth=0.5)
ax.axhline(y=0, color='k', lw=0.8)
ax.set_xlabel('Eigenvalue index', fontsize=11)
ax.set_ylabel('$\\lambda_i$', fontsize=11)
ax.set_title(f'Hessian eigenvalues at best particle (index-1: {is_idx1})', fontsize=12)
# Annotate the zero modes
ax.annotate(f'{eig_gap} zero\nmodes', xy=(eig_gap/2 - 0.5, 0), fontsize=8,
           ha='center', va='bottom', color='gray')
plt.tight_layout()
plt.savefig('../data/eigenvalues_best.pdf', bbox_inches='tight', dpi=150)
plt.show()

print(f"Best particle index: {best_idx}")
print(f"Energy: {E_best:.6f}")
print(f"Is index-1: {is_idx1}")
print(f"Eigenvalues: {evals_np.round(4)}")

### Top-5 Highest-Weight Particles

Cluster configurations of the 5 particles with largest weights, displayed in the style of the paper's Tables 4-5.

In [ ]:
configs_top = [np.array(result_sspd['X_final'][i]) for i in top_indices]
energies_top = [float(V_fns['V'](result_sspd['X_final'][i])) for i in top_indices]
titles_top = [f'$n$={i}, E={e:.3f}\n$w$={weights[i]:.4f}' 
              for i, e in zip(top_indices, energies_top)]

fig = plot_cluster_grid(configs_top, titles_top, ncols=5, figsize=(16, 3.5))
fig.suptitle('Top-5 Highest-Weight SSPD Particles', fontsize=13, y=1.02)
plt.savefig('../data/top5_clusters.pdf', bbox_inches='tight', dpi=150)
plt.show()

# Check index-1 status for each
for i, idx in enumerate(top_indices):
    Xi = result_sspd['X_final'][idx]
    is_i1, ev_i = check_index1(Xi, hess_fn, V_fns['eig_gap'])
    print(f"  Particle {idx}: E={energies_top[i]:.4f}, w={weights[idx]:.4f}, "
          f"index-1={is_i1}, lambda_1={float(ev_i[0]):.4f}")